# 🦆 Microduck RL: Huấn luyện Vượt Địa hình Gồ ghề (Rough Terrain) trên Google Colab

Notebook này thiết lập môi trường hoàn chỉnh và chạy huấn luyện mô hình **Microduck Bipedal Robot** trên địa hình gồ ghề (`Mjlab-Velocity-Rough-MicroDuck`) với **2048 environments** song song trên GPU của Google Colab.

> **Các quy tắc chuẩn:**
> - Kích hoạt ma trận đối xứng 61D (`ENABLE_SYMMETRY=True`) loại bỏ triệt để hiện tượng lệch bước.
> - Tăng trọng số phạt trôi hướng `heading_hold` = 2.5.
> - Cấu hình solver contact chống nổ NaN: `nconmax=200`, `iterations=30`.
> - Nướng sẵn Running Observation Normalizer vào file ONNX khi export.

### Bước 1: Kiểm tra GPU & Clone mã nguồn mới nhất

In [ ]:
# Kiểm tra thông số GPU (khuyến nghị T4, L4 hoặc A100)
!nvidia-smi

# Clone repo Microduck
import os
if not os.path.exists("Microduck"):
    !git clone https://github.com/rootkind35-del/Microduck.git
%cd Microduck
!git checkout main
!git pull origin main

### Bước 2: Cài đặt công cụ và đồng bộ môi trường `uv`

In [ ]:
!pip install -q uv
!uv sync

### Bước 3: Đăng nhập Weights & Biases (Tùy chọn)
*(Nếu muốn theo dõi đồ thị realtime trên wandb.ai, hãy dán API key khi được hỏi. Nếu muốn chạy offline không cần tài khoản, chọn dòng WANDB_MODE=offline)*

In [ ]:
# Để đồng bộ đồ thị lên tài khoản WandB:
!uv run wandb login

# HOẶC nếu không muốn đăng nhập tài khoản wandb, bỏ comment dòng dưới:
# %env WANDB_MODE=offline

### Bước 4: Chạy SMOKE TEST (Quy tắc bắt buộc theo AGENTS.md)
Chạy 5 vòng lặp thử nghiệm ở 64 envs để xác nhận CUDA, MuJoCo Warp và hệ thống hàm phạt hoạt động trơn tru không lỗi trước khi chạy dài.

In [ ]:
!uv run train Mjlab-Velocity-Rough-MicroDuck --env.scene.num-envs 64 --agent.max_iterations 5

### Bước 5: Chạy Huấn luyện chính thức (2048 Envs trên Địa hình Gồ ghề)

In [ ]:
# Chạy huấn luyện với 2048 envs trên GPU Colab:
!uv run train Mjlab-Velocity-Rough-MicroDuck \
  --env.scene.num-envs 2048 \
  --agent.max_iterations 3000

### Bước 6: Xuất mô hình sang file ONNX (Baked Normalizer)

In [ ]:
# Thay thế <run_path> bằng run ID thực tế hiển thị trên wandb của bạn (ví dụ: your_entity/mjlab_microduck/abcdefgh)
# Hoặc xuất trực tiếp từ checkpoint trong thư mục logs/:
!uv run scripts/export.py Mjlab-Velocity-Rough-MicroDuck --wandb-run-path <entity>/mjlab_microduck/<run_id>

### Bước 7: Tải file mô hình ONNX về máy tính

In [ ]:
from google.colab import files
import glob

# Tìm file onnx vừa xuất
onnx_files = glob.glob("*.onnx") + glob.glob("logs/**/*.onnx", recursive=True)
if onnx_files:
    latest_onnx = max(onnx_files, key=os.path.getmtime)
    print(f"Đang tải file {latest_onnx} về máy...")
    files.download(latest_onnx)
else:
    print("Chưa tìm thấy file .onnx nào.")